# Files

`pathlib.Path` builds paths with `/`. The same code works on Windows, macOS
and Linux.

In [1]:
from pathlib import Path

data_dir = Path("data")
jokes_path = data_dir / "jokes.txt"
print(jokes_path.name)
print(jokes_path.suffix)
print(jokes_path.exists())

jokes.txt
.txt
True


## Reading a text file

Always give `encoding="utf-8"`: otherwise Windows may read "é" as "Ã©".

In [2]:
with open(jokes_path, encoding="utf-8") as file:
    text = file.read()
print(text)

I told my stage fright to calm down. It said it would, after the show.
The Grand Casino has 650 seats. Tonight, 649 of them laughed.
Improv rule number one: say yes. Rule number two: say yes again.
A lineup is like a train: nobody wants to be the last carriage.



In [3]:
with open(jokes_path, encoding="utf-8") as file:
    for line in file:
        print(len(line), line.strip())

71 I told my stage fright to calm down. It said it would, after the show.
62 The Grand Casino has 650 seats. Tonight, 649 of them laughed.
65 Improv rule number one: say yes. Rule number two: say yes again.
64 A lineup is like a train: nobody wants to be the last carriage.


`with` closes the file at the end of the block, even if an error happens.
Each line ends with `\n`: `strip()` removes it.

In [4]:
open(data_dir / "missing.txt", encoding="utf-8")

FileNotFoundError: [Errno 2] No such file or directory: 'data/missing.txt'

## Writing

In [5]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
notes_path = output_dir / "notes.txt"

with open(notes_path, "w", encoding="utf-8") as file:
    file.write("Nora Lebrun: invite again in 2028\n")
    file.write("Théâtre des Planches: check the lights\n")

with open(notes_path, "a", encoding="utf-8") as file:
    file.write("Le Kiosque: 120 seats\n")

print(notes_path.read_text(encoding="utf-8"))

Nora Lebrun: invite again in 2028
Théâtre des Planches: check the lights
Le Kiosque: 120 seats



`"w"` replaces the content, `"a"` adds at the end. `read_text()` and
`write_text()` open, read or write, and close in one call.

## CSV

In [6]:
csv_path = data_dir / "shows.csv"
print(csv_path.read_text(encoding="utf-8"))

title,venue,start,duration,price
Nothing Serious,Théâtre des Planches,21:00,75,22.0
Open Mic Night,Le Kiosque,22:30,90,0.0
Tall Tales,Grand Casino,20:30,80,28.0
Late Laughs,Le Kiosque,23:30,60,12.0
Brunch Show,Théâtre des Planches,11:00,60,15.0



In [7]:
import csv

with open(csv_path, encoding="utf-8", newline="") as file:
    for row in csv.reader(file):
        print(row)

['title', 'venue', 'start', 'duration', 'price']
['Nothing Serious', 'Théâtre des Planches', '21:00', '75', '22.0']
['Open Mic Night', 'Le Kiosque', '22:30', '90', '0.0']
['Tall Tales', 'Grand Casino', '20:30', '80', '28.0']
['Late Laughs', 'Le Kiosque', '23:30', '60', '12.0']
['Brunch Show', 'Théâtre des Planches', '11:00', '60', '15.0']


In [8]:
with open(csv_path, encoding="utf-8", newline="") as file:
    shows = list(csv.DictReader(file))
print(shows[0])

{'title': 'Nothing Serious', 'venue': 'Théâtre des Planches', 'start': '21:00', 'duration': '75', 'price': '22.0'}


In [9]:
print(shows[0]["duration"] + 10)

TypeError: can only concatenate str (not "int") to str

Everything read from a CSV file is text. Convert the numbers yourself:

In [10]:
for show in shows:
    show["duration"] = int(show["duration"])
    show["price"] = float(show["price"])
print(shows[0]["duration"] + 10)

85


In [11]:
short_path = output_dir / "short_shows.csv"
with open(short_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["title", "duration"])
    writer.writeheader()
    for show in shows:
        if show["duration"] <= 60:
            writer.writerow({"title": show["title"], "duration": show["duration"]})

print(short_path.read_text(encoding="utf-8"))

title,duration
Late Laughs,60
Brunch Show,60



`newline=""` avoids an empty line between rows on Windows. A CSV saved by
Excel in French uses `;`: `csv.reader(file, delimiter=";")`.

## JSON

JSON keeps numbers, lists and dicts as they are.

In [12]:
import json

json_path = output_dir / "shows.json"
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(shows, file, indent=2, ensure_ascii=False)

print(json_path.read_text(encoding="utf-8")[:180])

[
  {
    "title": "Nothing Serious",
    "venue": "Théâtre des Planches",
    "start": "21:00",
    "duration": 75,
    "price": 22.0
  },
  {
    "title": "Open Mic Night",
    "


In [13]:
with open(json_path, encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded == shows)
print(type(loaded[0]["duration"]))

True
<class 'int'>


`ensure_ascii=False` writes `Théâtre` instead of `Th\u00e9\u00e2tre`.

## A file next to the script

In a `.py` file, `Path("programme.json")` depends on the folder the terminal is
in. `__file__` is the path of the script itself:

```python
DATA_FILE = Path(__file__).parent / "programme.json"
```